# Phase 1 — Fine-tune **Gemma 2-2B** on Arabic WSD (Dataset A / El-Razzaz)

Reproduces the paper's Gemma fine-tuning result, scaled down from 9B → 2B so it fits a **free Colab T4**.

**Before you run:**
1. `Runtime → Change runtime type → GPU (T4)`.
2. Commit & push your **fixed** scripts to your GitHub fork, then set `REPO_URL` below to that fork.
3. (Optional) Add an `HF_TOKEN` Colab secret — not required for public `unsloth/...` weights.

This notebook just runs the repo's `finetuning.py` → `infer_model.py` → `eval.py` (single source of truth). Phase 2 is identical except the **Config** cell.

In [ ]:
# 0. Confirm we have a GPU
!nvidia-smi

In [ ]:
# 1. Install dependencies (Unsloth pulls a compatible torch/CUDA stack on Colab)
!pip install -q -U unsloth trl peft transformers datasets bitsandbytes accelerate scikit-learn python-dotenv

In [ ]:
# 2. Mount Google Drive (artifacts persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone YOUR fork (with the fixed scripts). Falls back to a clear error if not set.
REPO_URL = 'https://github.com/<your-username>/Arabic-WSD-LLM.git'  # <-- EDIT: your fork
REPO_DIR = '/content/Arabic-WSD-LLM'
BRANCH   = 'phase1-gemma2-2b-datasetA'  # branch with the fixed scripts (use 'main' if you merged)

import os, shutil
assert '<your-username>' not in REPO_URL, 'Set REPO_URL to your pushed fork first.'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
SCRIPT_DIR = os.path.join(REPO_DIR, 'Gemma', 'Fine-tuning', 'Dataset-A')
print('Scripts:', os.listdir(SCRIPT_DIR))

In [ ]:
# 4. Stage Dataset A into Drive at PROJECT_DIR/data (the scripts read from here)
PROJECT_DIR = '/content/drive/MyDrive/WSD_Project'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

src_ds = os.path.join(REPO_DIR, 'Datasets', 'Dataset-A')
for f in ['test_set.json','test_truth.json','test_dictionary.json',
          'train80_set.json','train80_truth.json','train80_dictionary.json']:
    shutil.copy(os.path.join(src_ds, f), os.path.join(DATA_DIR, f))
shutil.copy(os.path.join(SCRIPT_DIR, 'fine_tuning_dataset_elrazzaz.jsonl'),
            os.path.join(DATA_DIR, 'fine_tuning_dataset_elrazzaz.jsonl'))
print('Staged:', os.listdir(DATA_DIR))

## Config — the only cell that differs between Phase 1 and Phase 2

In [ ]:
os.environ['WSD_PROJECT_DIR'] = PROJECT_DIR
os.environ['WSD_BASE_MODEL']  = 'unsloth/gemma-2-2b'   # Phase 1 base model
os.environ['WSD_MODEL_TAG']   = 'gemma2_2b'            # names the output folder
os.environ['WSD_MAX_SEQ_LEN'] = '1024'
# Optional HF token (only needed for gated weights):
# from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Config set for', os.environ['WSD_MODEL_TAG'])

In [ ]:
# 5. Fine-tune (LoRA, 4-bit). ~3 epochs over ~9.9k examples; saves merged_16bit to Drive.
!python {SCRIPT_DIR}/finetuning.py

In [ ]:
# 6. Inference over the 3,110-sentence test set → predictions JSON + debug log
!python {SCRIPT_DIR}/infer_model.py

In [ ]:
# 7. Evaluate → accuracy + macro-F1 report JSON
!python {SCRIPT_DIR}/eval.py

In [ ]:
# 8. Sanity check: show the report and a few raw predictions
import json
tag = os.environ['WSD_MODEL_TAG']
out = os.path.join(PROJECT_DIR, 'outputs', tag)
print('REPORT:', json.dumps(json.load(open(os.path.join(out, f'report_{tag}.json'))), indent=2))
print('\n--- first 1500 chars of debug log ---')
print(open(os.path.join(out, f'debug_{tag}.txt'), encoding='utf-8').read()[:1500])